# GNNs on Molecular Solubility (ESOL)

In this exercise you'll predict molecular solubility (log mol/L) from molecular graphs using Graph Neural Networks. We'll compare four model variants:

1. **1 GCN layer + mean pooling** (provided)
2. **2 GCN layers + mean pooling**
3. **3 GCN layers + mean pooling**
4. **3 GCN layers + attentional pooling**


## Setup

In [ ]:
# If pytorch geometric (a separate library from pytorch) is not installed, uncomment:
# !pip install torch_geometric networkx

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import MoleculeNet
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.nn.aggr import AttentionalAggregation
from torch_geometric.utils import to_networkx
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import tqdm

torch.manual_seed(47)
np.random.seed(47)

# Device: CUDA -> MPS -> CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
    
print(f'Using device: {device}')

### Load ESOL and build graphs

ESOL is a small dataset (~1128 molecules) of measured aqueous solubility values. It is available through torch geometric via `MoleculeNet` which constructs the molecular graphs for us. Each molecule becomes a `Data` object with:
- `x`: node (atom) features, shape `[num_atoms, num_node_features]`
- `edge_index`: bond connectivity, shape `[2, num_edges]`
- `y`: target solubility, shape `[1]`

In [ ]:
dataset = MoleculeNet(root='data/ESOL', name='ESOL')
dataset = dataset.shuffle()

print(f'Dataset: {dataset}')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of node features: {dataset.num_node_features}')
print(f'Example graph: {dataset[0]}')

print("\nNodes:")
print(dataset[0].x)
print("\nBonds:")
print(dataset[0].edge_index)


### Visualize 4 molecules

Let's see what these graphs actually look like. Nodes are atoms (labeled with atomic number from the first node-feature column), edges are bonds.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, idx in zip(axes.flatten(), range(4)):
    data = dataset[idx]
    G = to_networkx(data, to_undirected=True)

    # PyG MoleculeNet stores atomic number in the first column of x
    atomic_numbers = data.x[:, 0].tolist()
    labels = {i: z for i, z in enumerate(atomic_numbers)}

    pos = nx.spring_layout(G, seed=47)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=400)
    nx.draw_networkx_edges(G, pos, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=9)

    title = f'Molecule {idx}  |  log-solubility = {data.y.item():.2f}'
    if hasattr(data, 'smiles') and data.smiles is not None:
        title += f'\n{data.smiles}'
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 80/10/10 split
n = len(dataset)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_dataset = dataset[:n_train]
val_dataset = dataset[n_train:n_train + n_val]
test_dataset = dataset[n_train + n_val:]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

### Training utilities

Same structure as the standard `train_model` from earlier discussions, adapted for torch geometric batches. torch geometric's DataLoader yields `Batch` objects (not `(x, y)` tuples), so we unpack `batch.x`, `batch.edge_index`, `batch.batch`, and `batch.y` inside the loop.

In [ ]:
def train_model(model, train_loader, val_loader, loss_fn, optimizer,
                epochs=100, print_every=10):

    history = {'train': [], 'val': []}

    for epoch in tqdm.tqdm(range(epochs)):

        # ── Training phase ──
        model.train()
        epoch_train_loss = 0.0

        for batch in train_loader:
            batch = batch.to(device)
            x = batch.x.float()
            y = batch.y.view(-1).float()

            predictions = model(x, batch.edge_index, batch.batch)
            loss = loss_fn(predictions, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / len(train_loader)

        # ── Validation phase ──
        model.eval()
        epoch_val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                x = batch.x.float()
                y = batch.y.view(-1).float()

                predictions = model(x, batch.edge_index, batch.batch)
                loss = loss_fn(predictions, y)
                epoch_val_loss += loss.item()

        avg_val_loss = epoch_val_loss / len(val_loader)

        # Record
        history['train'].append(avg_train_loss)
        history['val'].append(avg_val_loss)

        if (epoch + 1) % print_every == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}  |  "
                  f"Train Loss: {avg_train_loss:.4f}  |  "
                  f"Val Loss: {avg_val_loss:.4f}")

    return history


def plot_losses(history):
    plt.figure(figsize=(8, 5))
    plt.plot(history['train'], label='Train Loss', linewidth=2)
    plt.plot(history['val'], label='Validation Loss',
             linewidth=2, linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate_test_rmse(model, loader):
    """Compute RMSE on a held-out loader."""
    model.eval()
    total_sq_err = 0.0
    n_samples = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            x = batch.x.float()
            y = batch.y.view(-1).float()
            pred = model(x, batch.edge_index, batch.batch)
            total_sq_err += ((pred - y) ** 2).sum().item()
            n_samples += y.numel()
    return np.sqrt(total_sq_err / n_samples)

## Reference model: 1 layer + mean pooling

Here's the reference implementation. Read it carefully — your job is to write three variants of this.

In [ ]:
HIDDEN_DIM = 64

class GNN1Mean(nn.Module):
    def __init__(self, num_features, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, batch):
        # Message passing
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        
        # Pooling
        x = global_mean_pool(x, batch)

        return self.head(x).view(-1)

In [ ]:
print('Training: 1 layer + mean pooling')
model_1mean = GNN1Mean(dataset.num_node_features).to(device)
optimizer = torch.optim.Adam(model_1mean.parameters(), lr=3e-3)
loss_fn = nn.MSELoss()

history_1mean = train_model(model_1mean, train_loader, val_loader,
                            loss_fn, optimizer, epochs=100, print_every=20)
plot_losses(history_1mean)
rmse_1mean = evaluate_test_rmse(model_1mean, test_loader)
print(f'Test RMSE: {rmse_1mean}')

## Your turn

Implement the three variants below. Each one is a small modification of `GNN1Mean`.

1. 2 GNN layers + mean pooling
2. 3 GNN layers + mean pooling
3. 3 GNN layers + attentional pooling

**Tips:**
- Apply ReLU between conv layers, just like in the reference.
- The signature of `forward` should stay the same.
- For the attentional pooling variant, replace `global_mean_pool` with `AttentionalAggregation`. The key difference: instead of averaging node embeddings uniformly, the model learns a per-node attention score and computes a weighted sum.Look up the `AttentionalAggregation` constructor signature. it needs at least one argument.

## Discussion

1. Did adding depth help? At what point did it stop helping (or start hurting)?
2. Did attentional pooling beat mean pooling at the same depth?
3. Look at the train/val loss curves, does any variant show signs of overfitting? Of underfitting?
4. Run the notebook again. How stable are these rankings across runs?